## Imports and db setup

In [1]:
import sys
from pathlib import Path
import duckdb
import zipfile
import os
import pandas as pd



data_dir = Path("../data").resolve()
assert data_dir.is_dir(), f"Data folder not found: {data_dir}"

con = duckdb.connect(str(data_dir / "project.duckdb"))

print("Database:", data_dir / "project.duckdb")
print(con.sql("SELECT version()").fetchall()[0][0])

Database: C:\Users\marks\.workspace\projects\data4600-project\code\data\project.duckdb
v2.0.0


## Add AK-2025 data to DB

In [2]:
STATES = (
    "AZ CA CO DC FL GA IL IN KS LA MD MA MI MN MO "
    "NV NJ NY NC OH PA SC TN TX WA WI"   # WI: in SCOPE.md, missing from download_data.py, confirm with Manthan
).split()

tables = [
    ("NIBRS_incident.csv", "incident"),
    ("agencies.csv", "agencies"),
    ("NIBRS_OFFENSE.csv", "offense"),
    ("NIBRS_OFFENDER.csv", "offender"),
    ("NIBRS_VICTIM.csv", "victim"),
    ("NIBRS_VICTIM_OFFENSE.csv", "victim_offense"),
    ("NIBRS_VICTIM_OFFENDER_REL.csv", "victim_offender_rel")
]

log = []

def table_exists(table_name):
    return con.sql(f"""
        SELECT count(*) FROM information_schema.tables
        WHERE table_name = '{table_name}'
    """).fetchall()[0][0] > 0

def add_missing_columns(table_name, csv_path):
    csv_cols = [r[0] for r in con.sql(f"DESCRIBE SELECT * FROM read_csv('{csv_path}', all_varchar=true)").fetchall()]
    table_cols = {r[0].lower() for r in con.sql(f"DESCRIBE {table_name}").fetchall()}
    for col in csv_cols:
        if col.lower() not in table_cols:
            con.sql(f'ALTER TABLE {table_name} ADD COLUMN "{col}" VARCHAR')
            print(f"  added column {table_name}.{col}")

def load_state_year(state, year, reload=False):
    zip_path = data_dir / "nibrs" / str(year) / f"{state}-{year}.zip"
    extracted_path = data_dir / "extracted" / str(year) / state

    if not zip_path.exists():
        log.append({"state": state, "year": year, "table": None, "rows": None, "status": "zip missing", "error": None})
        print(f"{state}-{year}: zip missing")
        return

    with zipfile.ZipFile(zip_path) as zipped:
        for csv_name, table_name in tables:
            entry = {"state": state, "year": year, "table": table_name, "rows": None, "status": None, "error": None}
            try:
                matches = [
                    name for name in zipped.namelist()
                    if name.split("/")[-1].lower() == csv_name.lower()
                ]
                if not matches:
                    entry["status"] = "csv missing"
                    continue

                if not reload and table_exists(table_name):
                    already = con.sql(f"""
                        SELECT count(*) FROM {table_name}
                        WHERE source_state = '{state}' AND source_year = {year}
                    """).fetchall()[0][0]
                    if already > 0:
                        entry["status"], entry["rows"] = "skipped", already
                        continue

                csv_path = zipped.extract(matches[0], extracted_path)

                con.sql(f"""
                    CREATE TABLE IF NOT EXISTS {table_name} AS
                    SELECT *, '{state}' AS source_state, {year} AS source_year
                    FROM read_csv('{csv_path}', all_varchar=true)
                    LIMIT 0
                """)
                add_missing_columns(table_name, csv_path)
                con.sql(f"""
                    DELETE FROM {table_name}
                    WHERE source_state = '{state}' AND source_year = {year}
                """)
                con.sql(f"""
                    INSERT INTO {table_name} BY NAME
                    SELECT *, '{state}' AS source_state, {year} AS source_year
                    FROM read_csv('{csv_path}', all_varchar=true)
                """)
                db_rows = con.sql(f"""
                    SELECT count(*) FROM {table_name}
                    WHERE source_state = '{state}' AND source_year = {year}
                """).fetchall()[0][0]
                csv_rows = con.sql(f"""
                    SELECT count(*) FROM read_csv('{csv_path}', all_varchar=true)
                """).fetchall()[0][0]
                assert db_rows == csv_rows, f"table {db_rows}, CSV {csv_rows}"

                entry["status"] = "loaded" if db_rows else "empty"
                entry["rows"] = db_rows
                os.remove(csv_path)
            except Exception as e:
                entry["status"], entry["error"] = "failed", str(e)[:300]
            finally:
                log.append(entry)
                print(f"{state}-{year} {table_name}: {entry['status']} {entry['rows'] or ''}")


In [3]:
for year in range(2025, 2015, -1):        # 2025 back to 2016
    for state in STATES:
        load_state_year(state, year)

log_df = pd.DataFrame(log)
con.sql("CREATE OR REPLACE TABLE load_log AS SELECT * FROM log_df")
log_df["status"].value_counts()

AZ-2025 incident: skipped 225182
AZ-2025 agencies: skipped 95
AZ-2025 offense: skipped 262933
AZ-2025 offender: skipped 252851
AZ-2025 victim: skipped 259574
AZ-2025 victim_offense: skipped 287365
AZ-2025 victim_offender_rel: skipped 128257
CA-2025 incident: skipped 1254218
CA-2025 agencies: skipped 727
CA-2025 offense: skipped 1437518
CA-2025 offender: skipped 1415485
CA-2025 victim: skipped 1423295
CA-2025 victim_offense: skipped 1560107
CA-2025 victim_offender_rel: skipped 703987
CO-2025 incident: skipped 260947
CO-2025 agencies: skipped 229
CO-2025 offense: skipped 302758
CO-2025 offender: skipped 294159
CO-2025 victim: skipped 298669
CO-2025 victim_offense: skipped 332463
CO-2025 victim_offender_rel: skipped 138647
DC-2025 incident: skipped 58541
DC-2025 agencies: skipped 2
DC-2025 offense: skipped 67511
DC-2025 offender: skipped 69346
DC-2025 victim: skipped 67665
DC-2025 victim_offense: skipped 74091
DC-2025 victim_offender_rel: skipped 41742
FL-2025 incident: skipped 448935
FL-

KeyboardInterrupt: 

## Add lookup tables to db

In [ ]:
lookup_state, lookup_year = "AZ", 2025

zip_path = data_dir / "nibrs" / str(lookup_year) / f"{lookup_state}-{lookup_year}.zip"
extracted_path = data_dir / "extracted" / str(lookup_year) / lookup_state

lookup_tables = [
    ("NIBRS_OFFENSE_TYPE.csv", "offense_type"),
    ("NIBRS_RELATIONSHIP.csv", "relationship"),
    ("NIBRS_VICTIM_TYPE.csv", "victim_type")
]

with zipfile.ZipFile(zip_path) as zipped:
    for csv_name, table_name in lookup_tables:
        matches = [
            name for name in zipped.namelist()
            if name.split("/")[-1].lower() == csv_name.lower()
        ]
        csv_path = zipped.extract(matches[0], extracted_path)
        con.sql(f"""
            CREATE OR REPLACE TABLE {table_name} AS
            SELECT *
            FROM read_csv('{csv_path}', all_varchar=true)
        """)
        db_rows = con.sql(f"SELECT count(*) FROM {table_name}").fetchall()[0][0]
        csv_rows = con.sql(f"SELECT count(*) FROM read_csv('{csv_path}', all_varchar=true)").fetchall()[0][0]
        assert db_rows == csv_rows, f"{table_name}: table {db_rows}, CSV {csv_rows}"
        print(f"{table_name}: {db_rows} rows")


offense_type: 86 rows
relationship: 30 rows
victim_type: 9 rows


## Duplicate checks

In [ ]:
dupe_checks = [
    ("incident", "incident_id"),
    ("agencies", "agency_id, source_state, source_year"),
    ("offense", "offense_id"),
    ("offender", "offender_id"),
    ("victim", "victim_id"),
    ("victim_offense", "victim_id, offense_id"),
    ("victim_offender_rel", "victim_id, offender_id"),
]

for table_name, key in dupe_checks:
    dupes = con.sql(f"""
        SELECT {key}, count(*)
        FROM {table_name}
        GROUP BY {key}
        HAVING count(*) > 1
    """).fetchall()
    assert dupes == [], f"{table_name}: duplicate {key}: {dupes[:5]}"
    print(f"{table_name}: no duplicate {key}")


incident: no duplicate incident_id
agencies: no duplicate agency_id, source_state, source_year
offense: no duplicate offense_id
offender: no duplicate offender_id
victim: no duplicate victim_id
victim_offense: no duplicate victim_id, offense_id
victim_offender_rel: no duplicate victim_id, offender_id


In [ ]:
# Sanity check
display(con.sql("""
    SELECT source_state, source_year, count(*) AS incidents
    FROM incident
    GROUP BY ALL
    ORDER BY source_state, source_year
""").fetchall())

[('AK', 2025, 14717),
 ('AL', 2025, 223728),
 ('AR', 2025, 141904),
 ('AZ', 2025, 225182),
 ('CA', 2025, 1254218),
 ('CO', 2025, 260947),
 ('CT', 2025, 93790),
 ('DC', 2025, 58541),
 ('DE', 2025, 47042),
 ('FL', 2025, 448935),
 ('GA', 2025, 366593),
 ('HI', 2025, 46776),
 ('IA', 2025, 104691),
 ('ID', 2025, 51571),
 ('IL', 2025, 503399),
 ('IN', 2025, 186210),
 ('KS', 2025, 143885),
 ('KY', 2025, 139937),
 ('LA', 2025, 159395),
 ('MA', 2025, 184982),
 ('MD', 2025, 239443),
 ('ME', 2025, 34071),
 ('MI', 2025, 374023),
 ('MN', 2025, 182092),
 ('MO', 2025, 238694),
 ('MS', 2025, 95762),
 ('MT', 2025, 38486),
 ('NC', 2025, 466821),
 ('ND', 2025, 34017),
 ('NE', 2025, 76388),
 ('NH', 2025, 40260),
 ('NJ', 2025, 214928),
 ('NM', 2025, 114544),
 ('NV', 2025, 165418),
 ('NY', 2025, 686932),
 ('OH', 2025, 387114),
 ('OK', 2025, 194379),
 ('OR', 2025, 175174),
 ('PA', 2025, 282523),
 ('RI', 2025, 28272),
 ('SC', 2025, 249856),
 ('SD', 2025, 37881),
 ('TN', 2025, 340672),
 ('TX', 2025, 1368545),


## Join checks

In [ ]:
same_source = "AND c.source_state = p.source_state AND c.source_year = p.source_year"

join_checks = [
    # (child table, column, parent table, column, also match state/year?)
    ("offender", "incident_id", "incident", "incident_id", True),
    ("victim", "incident_id", "incident", "incident_id", True),
    ("offense", "incident_id", "incident", "incident_id", True),
    ("incident", "agency_id", "agencies", "agency_id", True),
    ("victim_offense", "victim_id", "victim", "victim_id", True),
    ("victim_offense", "offense_id", "offense", "offense_id", True),
    ("victim_offender_rel", "victim_id", "victim", "victim_id", True),
    ("victim_offender_rel", "offender_id", "offender", "offender_id", True),
    ("offense", "offense_code", "offense_type", "offense_code", False),
    ("victim", "victim_type_id", "victim_type", "victim_type_id", False),
    ("victim_offender_rel", "relationship_id", "relationship", "relationship_id", False),
]

for child, child_col, parent, parent_col, match_source in join_checks:
    extra = same_source if match_source else ""
    orphans = con.sql(f"""
        SELECT count(*)
        FROM {child} c
        LEFT JOIN {parent} p
            ON c.{child_col} = p.{parent_col} {extra}
        WHERE c.{child_col} IS NOT NULL
            AND p.{parent_col} IS NULL
    """).fetchall()[0][0]
    
    if orphans and (child, child_col) == ("incident", "agency_id"):
        print(f"{child}.{child_col} -> {parent}: {orphans} incidents with agency not in agencies.csv (known FBI gap)")
        continue
    assert orphans == 0, f"{child}.{child_col}: {orphans} rows not found in {parent}"
    print(f"{child}.{child_col} -> {parent}: ok")


offender.incident_id -> incident: ok
victim.incident_id -> incident: ok
offense.incident_id -> incident: ok
incident.agency_id -> agencies: 459 incidents with agency not in agencies.csv (known FBI gap)
victim_offense.victim_id -> victim: ok
victim_offense.offense_id -> offense: ok
victim_offender_rel.victim_id -> victim: ok
victim_offender_rel.offender_id -> offender: ok
offense.offense_code -> offense_type: ok
victim.victim_type_id -> victim_type: ok
victim_offender_rel.relationship_id -> relationship: ok
